# P04 — Leave-one-out sensitivity and winsorisation panel

Computes (a) the leave-one-out sensitivity scores (LOO_max, τ₁, top-driver mode) per condition × seed, and (b) the winsorisation panel (raw vs winsorised Pearson under cell-type-specific p5/p95 leverage clipping).

**Manuscript reference:** Methods §2.6 (LOO); Results §3.2, §3.3, §3.4, §3.5; Figure 1 panels B and C; Table 1; Appendix A.

**Inputs:** the same per-seed severity detail h5ads consumed by P03, plus the cell-type-specific full severity references for the winsorisation thresholds.

**Outputs:**
- `data/eval/diag_loo_sensitivity_n100.csv` — per-seed LOO scores (CPA n=100 + GEARS n=7), matches the format consumed by the figure notebooks.
- `data/eval/diag_winsorise_n100.csv` — per-seed raw vs winsorised Pearson (CPA only).
- `data/eval/diag_winsorise_n100_summary.csv` — per-condition aggregate (4 rows).
- `data/eval/stage5_comparison_n100.csv` — Table 1 summary (per-condition mechanism mode and counts).


In [ ]:
import sys
from collections import Counter
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import pearsonr

sys.path.insert(0, str(Path.cwd()))
from perturb_style import DATA_DIR, PRECOMPUTED_DIR


## Configuration

In [ ]:
PREDICTIONS_DIR = DATA_DIR / "predictions" / "severity_details"
SEVERITY_REFS = {
    "K562": DATA_DIR / "severity_refs" / "replogle_K562_severity.csv",
    "RPE1": DATA_DIR / "severity_refs" / "replogle_RPE1_severity.csv",
}
EVAL_OUT_DIR = DATA_DIR / "eval"
EVAL_OUT_DIR.mkdir(exist_ok=True, parents=True)

CPA_SEEDS = list(range(1, 101))
GEARS_SEEDS = list(range(1, 8))
CONDITIONS = [(c, s) for c in ("K562", "RPE1") for s in ("random", "stratified")]

DRIVER_THRESH = 0.10        # threshold for driver_count_above_0.10
LOO_MAX_THRESH = 0.15       # operational single-driver threshold (Methods §2.6)
LOW_LOO_THRESH = 0.10       # below which "distributed" is considered (Methods §2.6)
RECURRENCE_FRAC = 0.30      # mode-driver recurrence threshold for single-driver


## Helpers

In [ ]:
def severity_h5ad_path(model: str, cell_type: str, split_type: str, seed: int) -> Path:
    fname = f"{model}_{cell_type}_{split_type}_seed{seed}.severity.h5ad"
    return PREDICTIONS_DIR / fname

def panel_thresholds(cell_type: str) -> tuple[float, float, int]:
    """(p5, p95, n_reference_genes) from the full cell-type-specific severity reference."""
    sev = pd.read_csv(SEVERITY_REFS[cell_type])
    lev = sev["leverage_score"].dropna().to_numpy(float)
    return float(np.percentile(lev, 5)), float(np.percentile(lev, 95)), int(len(lev))

def loo_scores(mag: np.ndarray, lev: np.ndarray, perts: np.ndarray | None) -> dict:
    """LOO sensitivity scores for one holdout. Returns dict with severity_pearson_raw,
    LOO_max_abs_delta, LOO_top3_sum_abs_delta, driver_count_above_0.10, top1_fraction,
    and (if perts is provided) top_loo_driver."""
    raw = pearsonr(mag, lev)[0]
    n = len(mag)
    absdelta = np.empty(n)
    for i in range(n):
        keep = np.arange(n) != i
        absdelta[i] = abs(raw - pearsonr(mag[keep], lev[keep])[0])
    order_idx = np.argsort(absdelta)[::-1]
    loo_max = float(absdelta[order_idx[0]])
    top3 = float(absdelta[order_idx[:3]].sum())
    driver_count = int((absdelta > DRIVER_THRESH).sum())
    top1_frac = float(loo_max / top3) if top3 > 0 else float("nan")
    out = {
        "severity_pearson_raw": round(float(raw), 4),
        "LOO_max_abs_delta": round(loo_max, 4),
        "LOO_top3_sum_abs_delta": round(top3, 4),
        "driver_count_above_0.10": driver_count,
        "top1_fraction": round(top1_frac, 4),
    }
    if perts is not None:
        out["top_loo_driver"] = str(np.asarray(perts)[order_idx[0]])
    return out

def _mad(x: np.ndarray) -> float:
    x = np.asarray(x, float)
    return float(np.median(np.abs(x - np.median(x))))

def _pct_change(raw_v: float, wins_v: float) -> float:
    if raw_v == 0:
        return float("nan")
    return (wins_v - raw_v) / abs(raw_v) * 100.0


## Run the analysis

If `data/predictions/severity_details/` does not exist, this notebook prints a helpful message and exits without writing output (same fallback as P03).

In [ ]:
if not PREDICTIONS_DIR.exists() or not any(PREDICTIONS_DIR.iterdir()):
    print(f"Predictions directory not found or empty: {PREDICTIONS_DIR}")
    print("This notebook produces data/eval/diag_loo_sensitivity_n100.csv,")
    print("data/eval/diag_winsorise_n100.csv, data/eval/diag_winsorise_n100_summary.csv,")
    print("and data/eval/stage5_comparison_n100.csv from per-seed prediction h5ads.")
    print()
    print("To produce predictions:")
    print("  ./run_all.sh --demo        # 2-seed verification (~30 min)")
    print("  ./run_all.sh --train-full  # full n=100 CPA + matched-n GEARS (~13 hours)")
    print()
    print("To skip and use the shipped precomputed CSVs:")
    print("  ./run_all.sh --figures")
    print()
    print(f"Existing precomputed CSVs: {PRECOMPUTED_DIR / 'eval'}")
    print("Exiting without writing output.")
else:
    import anndata as ad

    # ---- Pass 1: LOO sensitivity per (model, condition, seed) ----
    print("=== Computing LOO sensitivity scores ===")
    loo_rows = []
    for model, seeds in [("CPA", CPA_SEEDS), ("GEARS", GEARS_SEEDS)]:
        for cell, split in CONDITIONS:
            for seed in seeds:
                path = severity_h5ad_path(model, cell, split, seed)
                if not path.exists():
                    continue
                obs = ad.read_h5ad(path).obs
                mag = obs["predicted_mean_abs_delta"].to_numpy(float)
                lev = obs["leverage_score"].to_numpy(float)
                perts = obs["perturbation_target"].to_numpy() if "perturbation_target" in obs.columns else None
                scores = loo_scores(mag, lev, perts)
                loo_rows.append({"model_id": model, "cell_type": cell,
                                 "split_type": split, "seed": seed, **scores})
    loo_df = pd.DataFrame(loo_rows)
    loo_out = EVAL_OUT_DIR / "diag_loo_sensitivity_n100.csv"
    loo_df.to_csv(loo_out, index=False)
    print(f"Wrote {loo_out} ({len(loo_df)} rows)")

    # ---- Pass 2: winsorisation panel (CPA only) ----
    print("\n=== Computing winsorisation panel ===")
    refs = {c: panel_thresholds(c) for c in ("K562", "RPE1")}
    for c, (p5, p95, n) in refs.items():
        print(f"  {c}: p5={p5:.4f}  p95={p95:.4f}  n_reference_genes={n}")

    wins_rows = []
    for cell, split in CONDITIONS:
        p5, p95, _ = refs[cell]
        for seed in CPA_SEEDS:
            path = severity_h5ad_path("CPA", cell, split, seed)
            if not path.exists():
                continue
            obs = ad.read_h5ad(path).obs
            mag = obs["predicted_mean_abs_delta"].to_numpy(float)
            lev = obs["leverage_score"].to_numpy(float)
            lev_c = np.clip(lev, p5, p95)
            wins_rows.append({
                "condition": f"{cell} {split}",
                "cell_type": cell, "split_type": split, "seed": seed,
                "r_raw": float(pearsonr(mag, lev)[0]),
                "r_wins": float(pearsonr(mag, lev_c)[0]),
                "n_perts_capped": int(((lev < p5) | (lev > p95)).sum()),
            })
    wins_df = pd.DataFrame(wins_rows)
    wins_out = EVAL_OUT_DIR / "diag_winsorise_n100.csv"
    wins_df.to_csv(wins_out, index=False)

    # ---- Aggregate winsorisation summary (4 rows) ----
    summ_rows = []
    for cell, split in CONDITIONS:
        g = wins_df[(wins_df.cell_type == cell) & (wins_df.split_type == split)]
        if g.empty:
            continue
        raw, wins = g["r_raw"].to_numpy(float), g["r_wins"].to_numpy(float)
        rr, wr = float(raw.max() - raw.min()), float(wins.max() - wins.min())
        ram, wam = _mad(raw), _mad(wins)
        rw = float(np.percentile(raw, 95) - np.percentile(raw, 5))
        ww = float(np.percentile(wins, 95) - np.percentile(wins, 5))
        summ_rows.append({
            "condition": f"{cell} {split}",
            "n_present": int(len(g)),
            "raw_median": round(float(np.median(raw)), 4),
            "wins_median": round(float(np.median(wins)), 4),
            "raw_range": round(rr, 4),
            "wins_range": round(wr, 4),
            "range_pct_change": round(_pct_change(rr, wr), 1),
            "raw_MAD": round(ram, 4),
            "wins_MAD": round(wam, 4),
            "MAD_pct_change": round(_pct_change(ram, wam), 1),
            "raw_p5_p95_width": round(rw, 4),
            "wins_p5_p95_width": round(ww, 4),
            "p5_p95_pct_change": round(_pct_change(rw, ww), 1),
            "raw_sign_flips": int((raw < 0).sum()),
            "wins_sign_flips": int((wins < 0).sum()),
            "median_n_perts_capped": float(np.median(g["n_perts_capped"].to_numpy(float))),
        })
    wins_summary = pd.DataFrame(summ_rows)
    wins_summary_out = EVAL_OUT_DIR / "diag_winsorise_n100_summary.csv"
    wins_summary.to_csv(wins_summary_out, index=False)
    print(f"\nWrote {wins_out} ({len(wins_df)} rows)")
    print(f"Wrote {wins_summary_out} ({len(wins_summary)} rows)")

    # ---- Stage5 comparison (Table 1 source) ----
    print("\n=== Computing per-condition mechanism summary (Table 1 source) ===")
    cpa_loo = loo_df[loo_df.model_id == "CPA"]
    stage5_rows = []
    for cell, split in CONDITIONS:
        g = cpa_loo[(cpa_loo.cell_type == cell) & (cpa_loo.split_type == split)]
        if g.empty:
            continue
        sev = g["severity_pearson_raw"].to_numpy(float)
        high = g[g.LOO_max_abs_delta > LOO_MAX_THRESH]
        if "top_loo_driver" in g.columns and len(high) > 0:
            c = Counter(high["top_loo_driver"].dropna())
            mode_drv, mode_cnt = (c.most_common(1)[0] if c else (None, 0))
        else:
            mode_drv, mode_cnt = None, 0
        stage5_rows.append({
            "condition": f"{cell} {split}",
            "n": int(len(g)),
            "median": round(float(np.median(sev)), 4),
            "MAD": round(_mad(sev), 4),
            "IQR": round(float(np.percentile(sev, 75) - np.percentile(sev, 25)), 4),
            "p5_p95_width": round(float(np.percentile(sev, 95) - np.percentile(sev, 5)), 4),
            "sign_flips": int((sev < 0).sum()),
            "LOO_max_median": round(float(g.LOO_max_abs_delta.median()), 4),
            "top1_frac_median": round(float(g.top1_fraction.median()), 4),
            "n_LOOmax_gt_0.15": int((g.LOO_max_abs_delta > LOO_MAX_THRESH).sum()),
            "n_high_LOO_seeds": int(len(high)),
            "top_driver_mode": mode_drv,
            "top_driver_mode_count_among_high_LOO": mode_cnt,
        })
    stage5 = pd.DataFrame(stage5_rows)
    stage5_out = EVAL_OUT_DIR / "stage5_comparison_n100.csv"
    stage5.to_csv(stage5_out, index=False)
    print(f"Wrote {stage5_out} ({len(stage5)} rows)")
    print(stage5.to_string(index=False))
